In [ ]:
import scanpy as sc
import pandas as pd
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
import scrublet as scr
import tqdm.auto as tqdm
import gseapy as gp
import seaborn as sns
#import ktplotspy as kpy
import networkx as nx
#from bioinfokit.visuz import GeneExpression

import cellphonedb
#from cellphonedb.src.core.methods import cpdb_statistical_analysis_method
#from cellphonedb.src.core.methods import cpdb_analysis_method
#from cellphonedb.utils import search_utils

import itertools
from mousipy import translate

import re
import harmonypy

In [ ]:
%matplotlib  inline

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

# Read in Data

In [ ]:
import scanpy as sc
import anndata as ad

data_paths = [
    '/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sg18_soupx.h5ad',
    '/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sg20_soupx.h5ad',
    '/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sg24_soupx.h5ad',
    '/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sg26_soupx.h5ad',
    '/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sg28_soupx.h5ad'
]

adatas_in_memory = []

for file_path in data_paths:
    adata_backed = sc.read_h5ad(file_path, backed='r')

    # Load full data into memory for modifications
    adata_mem = adata_backed.to_memory()
    
    # Rename obs column
    if 'orig.ident' in adata_mem.obs.columns:
        adata_mem.obs.rename(columns={'orig.ident': 'sample'}, inplace=True)
    
    # Make unique names
    adata_mem.var_names_make_unique()
    adata_mem.obs_names_make_unique()
    
    adatas_in_memory.append(adata_mem)

# Concatenate all in-memory AnnData objects
adata = ad.concat(adatas_in_memory, join='outer', label='batch', keys=[f'file{i}' for i in range(len(adatas_in_memory))])

# Now you can proceed further
adata.obs = adata.obs.rename(columns={'orig.ident': 'sample'})

# Ensure unique cell and gene names
adata.var_names_make_unique()
adata.obs_names_make_unique()

In [ ]:
# Step 1: Create a dictionary for mapping old names to new names
rename_dict = {
    'sg18': 'ctrl',
    'sg20': 'age',
    'sg24': 'sAct',
    'sg26': 'DR',
    'sg28': 'combi'
}

# Step 2: Apply the mapping to adata.obs['sample']
adata.obs['sample'] = adata.obs['sample'].map(rename_dict)

# Scrublet

In [ ]:
results = pd.read_csv("/data/bonn-epyc/projects/manuela/aging/scRNA_aging-mouse/scrublet-aging_soupxed.csv", index_col = 0)

# Basic preprocessing

In [ ]:
sc.pp.filter_cells(adata, min_genes=400)
sc.pp.filter_genes(adata, min_cells=3)

adata.var['mt'] = adata.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], jitter =0.4, multi_panel=True)

sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
# instead of picking subjectively, you can use a quantile
upper_lim = np.quantile(adata.obs.n_genes_by_counts.values, .98)
lower_lim = np.quantile(adata.obs.n_genes_by_counts.values, .02)
print(f'{lower_lim} to {upper_lim}')

#adata = adata[(adata.obs.n_genes_by_counts < upper_lim) & (adata.obs.n_genes_by_counts > lower_lim)]

In [ ]:
adata = adata[adata.obs.n_genes < 5000, :]
adata = adata[adata.obs.pct_counts_mt < 5, :]

In [ ]:
# Normalize, log-transform and scale the data
sc.pp.normalize_total(adata, target_sum=1e4) #normalize every cell to 10000 UMI (10000 counts)
print(adata.X[0,:].sum()) #sum up all counts for cell 0 -> sums will be 10000 now

sc.pp.log1p(adata) #change to log counts

## Clustering

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata.var[adata.var.highly_variable]

## expressions: SLC5A, Qpct

In [ ]:
# Create a pattern to match any variation of SGLT
pattern = re.compile(r'SLC5A', re.IGNORECASE)

# Search for matches in adata.raw.var_names
matching_genes = [gene for gene in adata.var_names if pattern.search(gene)]

# Print the matching genes
print("Matching genes:", matching_genes)
#gene_of_interest = 'Slc5a2'
gene_of_interest = matching_genes
# Create the violin plot
sc.pl.violin(adata, keys=gene_of_interest, groupby='sample', rotation=90)
sc.pl.dotplot(adata, var_names=gene_of_interest, groupby='sample', standard_scale = 'var')
sc.pl.matrixplot(adata, var_names=gene_of_interest, groupby='sample', standard_scale = 'var')

In [ ]:
genes_of_interest = ['Qpct', 'Qpctl', 'Ccl2']  # Add Ccl2 to your gene list
matching_genes = [gene for gene in adata.var_names if gene in genes_of_interest]

print("Matching genes:", matching_genes)
gene_of_interest = matching_genes

sc.pl.violin(adata, keys=gene_of_interest, groupby='sample', rotation=90)
sc.pl.dotplot(adata, var_names=gene_of_interest, groupby='sample', standard_scale='var')
sc.pl.matrixplot(adata, var_names=gene_of_interest, groupby='sample', standard_scale='var')


# Check if both genes are in adata.raw.var_names (original full set)
print("Genes in adata.raw:", [gene for gene in adata.var_names if gene.lower().startswith('qpct')])

# Are both genes (Qpct, Qpctl) actually highly variable?
print("Is Qpctl highly variable?:", adata.var.loc['Qpctl', 'highly_variable'])
print("Is Qpct highly variable?:", adata.var.loc['Qpct', 'highly_variable'])

In [ ]:
adata.raw = adata # new slot adata.raw to save data before processing

# Ensure 'Qpctl' stays in var_names
adata.var.loc['Qpctl', 'highly_variable'] = True

# Filter highly variable genes
adata = adata[:, adata.var.highly_variable]

# Regress out effects of total counts per cell and the percentage of mitochondrial genes expressed. 
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])

# Scale the data to unit variance and zero mean
sc.pp.scale(adata, max_value=10)

# Perform PCA
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca(adata, color = 'sample')
sc.pl.pca_variance_ratio(adata, log=True)

In [ ]:
#print(adata.raw.var)
sc.pl.violin(adata, keys=['Qpctl', 'Qpct', 'Ccl2'], groupby='sample', use_raw=True)

In [ ]:
genes_of_interest = ['Qpct', 'Qpctl', 'Ccl2']  # Add Ccl2 to your gene list
matching_genes = [gene for gene in adata.var_names if gene in genes_of_interest]

print("Matching genes:", matching_genes)
gene_of_interest = matching_genes

sc.pl.violin(adata, keys=gene_of_interest, groupby='sample', rotation=90)
sc.pl.dotplot(adata, var_names=gene_of_interest, groupby='sample', standard_scale='var')
sc.pl.matrixplot(adata, var_names=gene_of_interest, groupby='sample', standard_scale='var')

In [ ]:
# Run Harmony
harmony_out = harmonypy.run_harmony(
    adata.obsm['X_pca'], 
    adata.obs, 
    'sample'
)

# Store directly into internal dict, bypassing anndata validation
adata.obsm._data['X_pca_harmony'] = np.array(harmony_out.Z_corr)

# Verify
print(adata.obsm['X_pca_harmony'].shape)  # should show (24419, 50)

In [ ]:
# Calculate neighborhood graph
#sc.pp.neighbors(adata, n_neighbors=10, n_pcs=20, use_rep="X_pca_harmony")
sc.pp.neighbors(adata, n_pcs=20, metric="cosine", n_neighbors=30, use_rep="X_pca_harmony")

# Clustering
sc.tl.leiden(adata, resolution=0.31)

# Compute UMAP
sc.tl.umap(adata)
# Visualization
sc.pl.umap(adata, color=['sample', 'leiden'])

In [ ]:
print(adata.obs['sample'].value_counts())
print(f'Total nuclei retained: {adata.n_obs}')

In [ ]:
print(adata.obs['leiden'].value_counts().sort_index())
print(f'Total clusters: {adata.obs["leiden"].nunique()}')

# Find markers -> Celltype Annotation

In [ ]:
# Rank geneshttps://jh.bonn-epyc.int.ims.bio/user/mpoets/lab/workspaces/auto-W/tree/data/projects/manuela/scRNA_aging-mouse/00_full_analysis.ipynb#Pseudo-Bulk
sc.tl.rank_genes_groups(adata, 'leiden', method='t-test')

# Plot the ranked genes
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False)

In [ ]:
# Rank genes using the t-test method and store the results in 'adata.uns' (if not done already)
sc.tl.rank_genes_groups(adata, 'leiden', method='t-test')

# Extract the results from 'adata.uns' and store them in a dataframe
result_dict = adata.uns['rank_genes_groups']

# Initialize empty lists to store the results for each cluster
cluster_list = []
gene_list = []
score_list = []
p_value_list = []
padj_value_list = []
lfc_list = [] 

# Iterate over the clusters and extract the results
for cluster in result_dict['names'].dtype.names:
    genes = result_dict['names'][cluster]
    scores = result_dict['scores'][cluster]
    p_values = result_dict['pvals'][cluster]
    padj_values = result_dict['pvals_adj'][cluster]
    lfc_values = result_dict['logfoldchanges'][cluster]
    
    # Append the results to the lists
    cluster_list.extend([cluster] * len(genes))
    gene_list.extend(genes)
    score_list.extend(scores)
    p_value_list.extend(p_values)
    padj_value_list.extend(padj_values)
    lfc_list.extend(lfc_values)

# Create the DataFrame
result_df = pd.DataFrame({
    'cluster': cluster_list,
    'gene': gene_list,
    'score': score_list,
    'p_value': p_value_list,
    'padj_value': padj_value_list,
    'lfc': lfc_list,
})

# Print the dataframe
print(result_df[result_df['p_value'] != 0])

In [ ]:
# label all cells with an individual gene
glis3_i = np.where(adata.raw.var_names == 'Glis3')[0][0]

#get counts?
glis = adata.raw.X.toarray()[:, glis3_i]

adata.obs['glis3'] = glis
adata.obs[adata.obs['glis3'] > 0]

sc.pl.violin(adata, ['Glis3'], groupby='leiden')
sc.pl.violin(adata, ['Glis3'], groupby='sample',)

## Top 10

In [ ]:
# Get top 10 marker genes for each cluster
marker_genes = {}
for group in adata.uns['rank_genes_groups']['names'].dtype.names:
    marker_genes[group] = adata.uns['rank_genes_groups']['names'][group][:10]

# Convert marker_genes to pandas DataFrame for easy visualization
marker_genes_df = pd.DataFrame(marker_genes)
print(marker_genes_df)

In [ ]:
marker_genes_dict = marker_genes_df.to_dict(orient='list')
print(marker_genes_dict)

In [ ]:
celltype_dict = {
    '0': 'PTC', # Proximal Tubular Cells
    '1': 'DCT', # Distal Convoluted Tubule Cells
    '2': 'DCT/CD', # Distal Convoluted Tubule/Collecting Duct Cells
    '3': 'Unknown1',
    '4': 'EC',
    '5': 'Stressed/Artifact',
    '6': 'Unknown3',
    '7': 'PODO', # Podocyte
    '8': 'IT/CD', # Ion Transport/Collecting Duct Cells
    '9': 'Neuronal',
    '10': 'Hematopoietic/Immune',
    '11': 'PT', #Proximal tubule cells
    '12': 'Detox/Metab', # Detoxification/Metabolism Cells (Possibly Hepatic Cells)
    '13': 'Unknown5',
    '14': 'T-cells', #yes # Immune T Cells
    '15': 'Neuronal2',
    '16': 'Proliferating', #yes
    '17': 'Adipocytes' #yes
}

In [ ]:
sc.pl.umap(adata, color=['celltype'], legend_loc='on data', frameon=False)

In [ ]:
#annotation sydney
annotation = {
    "0": "PT-1", 
    "1": "TAL-1",
    "2": "DCT/CNT",
    "3": "PC_CNT",
    "4.0": "EC-2",
    "4.1": "EC-3",  
    "4.2": "EC-4",  
    "4.3": "EC-1(gEC)",  
    "4.4": "EC-5",  
    "4.5": "EC-6",  
    "5": "TAL-2",
    "6": "FIB",
    "7": "PODO",
    "8": "IC-B",
    "9": "Undef",    
    "10": "IC-A",
    "11": "PT-2",
    "12.0": "Uro",
    "12.1": "Undef",
    "12.2": "ATL",
    "12.3": "PC",
    "13": "vSMC/MC",
    "14": "IMM",
    "15": "PECs?", # Parietal Epithelial Cells
    "16": "PROLIF",
    "17": "ADIPO"
}

# Assign cell type labels based on the leiden clustering results
adata.obs['celltype'] = [annotation[i] for i in adata.obs['leiden']]

## Check with canonical markers

In [ ]:
markers = {
    'PTC': ['Dab2', 'Slc4a4'],
    'DCT': ['Slc12a1', 'Erbb4'],
    'DCT/CD': ['Slc12a3', 'Wnk1'],
    'GC': ['Ptpro', 'Magi2'],
    'IT/CD': ['Slc26a4', 'Tmem117'],
    'Hematopoietic/Immune': ['Atp6v0d2', 'Adgrf5'],
    'T-cells': ['Arhgap15', 'Ptprc'],
    'Proliferating': ['Diaph3', 'Top2a'],
    'Adipocytes': ['Acaca', 'Fabp4']
}

In [ ]:
sc.pl.dotplot(adata, markers, groupby='celltype')

# Pseudo Bulk

## SGLT expression

In [ ]:
# Create a pattern to match any variation of SGLT
pattern = re.compile(r'SLC5A', re.IGNORECASE)

# Search for matches in adata.raw.var_names
matching_genes = [gene for gene in adata.var_names if pattern.search(gene)]

#gene_of_interest = 'Slc5a2'
gene_of_interest = matching_genes
# Create the violin plot
sc.pl.violin(adata, keys=gene_of_interest, groupby='sample', rotation=90)
sc.pl.dotplot(adata, var_names=gene_of_interest, groupby='sample', standard_scale = 'var')
sc.pl.matrixplot(adata, var_names=gene_of_interest, groupby='sample', standard_scale = 'var')

## DEGs

In [ ]:
adata.uns['log1p']['base'] = None
sc.tl.rank_genes_groups(adata, 'sample', method='t-test', reference ='age')# Now plot the results

In [ ]:
sc.pl.rank_genes_groups(adata, n_genes=10)
#sc.pl.rank_genes_groups_dotplot(adata,n_genes=10)
#sc.pl.rank_genes_groups_dotplot(adata, n_genes=4, values_to_plot="logfoldchanges", cmap='bwr', vmin=-4, vmax=4, min_logfoldchange=3, colorbar_title='log fold change')

In [ ]:
p = 0.05
log2fc = 0 

In [ ]:
deg = sc.get.rank_genes_groups_df(adata, group= None)
deg_sAct = sc.get.rank_genes_groups_df(adata, group= 'sAct')
deg_DR = sc.get.rank_genes_groups_df(adata, group= 'DR')
deg_combi = sc.get.rank_genes_groups_df(adata, group= 'combi')
# filtered only by pval
deg_fi = deg[deg['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_sig = deg[(deg['pvals_adj'] < p) & (deg['logfoldchanges'].abs() > log2fc)]

print(deg.shape)
print(deg_fi.shape)
print(deg_sig.shape)

### Split in conditions and up/down

In [ ]:
# Upregulated genes
deg_sAct_up = deg_sAct[(deg_sAct['pvals_adj'] < p) & (deg_sAct['logfoldchanges'] > 0)].sort_values('logfoldchanges', ascending=False)
deg_DR_up = deg_DR[(deg_DR['pvals_adj'] < p) & (deg_DR['logfoldchanges'] > 0)].sort_values('logfoldchanges', ascending=False)
deg_combi_up = deg_combi[(deg_combi['pvals_adj'] < p) & (deg_combi['logfoldchanges'] > 0)].sort_values('logfoldchanges', ascending=False)

# Downregulated genes
deg_sAct_dw = deg_sAct[(deg_sAct['pvals_adj'] < p) & (deg_sAct['logfoldchanges'] < 0)].sort_values('logfoldchanges', ascending=True)
deg_DR_dw = deg_DR[(deg_DR['pvals_adj'] < p) & (deg_DR['logfoldchanges'] < 0)].sort_values('logfoldchanges', ascending=True)
deg_combi_dw = deg_combi[(deg_combi['pvals_adj'] < p) & (deg_combi['logfoldchanges'] < 0)].sort_values('logfoldchanges', ascending=True)

### Write csv

### Visualization

In [ ]:
# Sort the DataFrame by pvals_adj in ascending order (smaller p-values first)
deg_sig_sorted = deg_sig.sort_values(by='pvals_adj')

# Select the top 50 enriched genes
top_genes_enr = deg_sig_sorted.head(50) 

In [ ]:
sc.pl.dotplot(adata, top_genes_enr.names, standard_scale = 'var', groupby='sample')

In [ ]:
# Sort the DataFrame by pvals_adj in ascending order (smaller p-values first)
deg_sig_sorted = deg_sig.sort_values(by='logfoldchanges', key=abs, ascending=False)

# Select the top 50 enriched genes
top_genes = deg_sig_sorted.head(50) 

sc.pl.dotplot(adata, top_genes.names, standard_scale = 'var', groupby='sample')

In [ ]:
# First, filter the dataframe to include only the genes of interest
filtered_df = deg_sig

# Now, create a bar plot
plt.figure(figsize=(50, 6))
# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors = ['green' if x >= 0 else 'red' for x in filtered_df['logfoldchanges']]
plt.bar(filtered_df['names'], filtered_df['logfoldchanges'], color=colors)
plt.xlabel('Genes')
plt.ylabel('Log-Fold Changes')
plt.title('Log-Fold Changes of Genes of Interest SG20 vs. SG18')
plt.xticks(rotation=90)  # Rotate x-axis labels for readability
plt.tight_layout()
plt.show()

# Visualize expression patterns of these mouse genes, e.g., a heatmap
sc.pl.heatmap(adata, var_names=deg_sig.names, groupby = 'sample', use_raw = True)
sc.pl.violin(adata, keys=deg_sig.names, groupby = 'sample', use_raw = True)
sc.pl.dotplot(adata, var_names=deg_sig.names, groupby = 'sample', use_raw = True)
sc.pl.dotplot(adata, var_names=deg_sig.names, groupby = 'sample', use_raw = True, standard_scale =  'group')
sc.pl.matrixplot(adata,deg_sig.names, 'sample', dendrogram=True, cmap='Blues', standard_scale='var', colorbar_title='column scaled\nexpression')

## Over-representation analysis (Enrichment) 

In [ ]:
from gseapy.scipalette import SciPalette

NbDr = SciPalette.create_colormap() 

In [ ]:
deg_sig_sg20 = deg_sig[deg_sig['group']=='sg20']
deg_sig_sg24 = deg_sig[deg_sig['group']=='sg24']
deg_sig_sg26 = deg_sig[deg_sig['group']=='sg26']
deg_sig_sg28 = deg_sig[deg_sig['group']=='sg28']
print(deg_sig)
# List of dataframe names
df_names = ['deg_sig_sg20', 'deg_sig_sg24', 'deg_sig_sg26', 'deg_sig_sg28']

### GO BP

In [ ]:
# Loop over the dataframes
for df_name in df_names:
    
    # Get the dataframe
    df = globals()[df_name]
    # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_sig_", "")

    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    #print(degs_up)
    
    sc.pl.DotPlot(adata, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} GO BP upregulated", cmap='BuGn', figsize=(300, 5)).savefig('./dotplot.pdf' ,format='pdf')
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['GO_Biological_Process_2021', 'KEGG_2019_Mouse'],
                        outdir=None)

    enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['GO_Biological_Process_2021', 'KEGG_2019_Mouse'],
                        outdir=None)
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]

    # dotplot
    gp.dotplot(enr_up.res2d, figsize=(3,5), title=f"{df_short_name} GO BP upregulated", cmap = plt.cm.autumn_r)
    plt.show()

    gp.dotplot(enr_dw.res2d,
               figsize=(3,5),
               title=f"{df_short_name} GO BP downregulated",
               cmap = plt.cm.winter_r,
               size=5)
    plt.show()

    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-GO_BP",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-GO_BP",
                    color = ['b','r'])
    plt.show()

# Downstream analyses

In [ ]:
ann_adata = sc.read("/data/projects/manuela/scRNA_aging-mouse/annotated-aging-soupxed.h5ad", var_names="gene_symbols")
#ann_adata.raw = ann_adata
ann_adata.uns['log1p']['base'] = None
ann_adata.var

## SGLT expression

In [ ]:
# Create a pattern to match any variation# of SGLT
pattern = re.compile(r'SLC5A', re.IGNORECASE)

# Search for matches in adata.raw.var_names
matching_genes = [gene for gene in adata.var_names if pattern.search(gene)]
print(matching_genes)
gene_of_interest = 'Slc5a2'
#gene_of_interest = matching_genes

# Create the violin plot
sc.pl.violin(adata, keys=gene_of_interest, groupby='sample', rotation=90)
sc.pl.dotplot(adata, var_names=gene_of_interest, groupby='sample', standard_scale = 'var')
sc.pl.matrixplot(adata, var_names=gene_of_interest, groupby='sample', standard_scale = 'var')

In [ ]:
### Tubular only

In [ ]:
# Define the cell types corresponding to tubular cells
tubular_cell_types = [
    "PT-1", 
    "PT-2", 
    "TAL-1", 
    "TAL-2", 
    "DCT/CNT", 
    "PC_CNT", 
    "IC-A", 
    "IC-B", 
    "PC", 
    "ATL"
]

# Subset the data
adata_tubular = ann_adata[ann_adata.obs['celltype'].isin(tubular_cell_types)].copy()

#gene_of_interest = 'Slc5a2'
gene_of_interest = matching_genes

# Create the violin plot
sc.pl.violin(adata_tubular, keys=gene_of_interest, groupby='sample', rotation=90)
sc.pl.dotplot(adata_tubular, var_names=gene_of_interest, groupby='sample', standard_scale = 'var')
sc.pl.matrixplot(adata_tubular, var_names=gene_of_interest, groupby='sample', standard_scale = 'var')

## Differential Expression Analysis (Reference SG18)

In [ ]:
# Create a place to store results
results_vs18 = {}
# Specify the cell types of interest
celltypes = ['PODO', 'EC-1(gEC)', 'vSMC/MC', 
                         #'PEC',
                         'PT-1', 'PT-2', 'IMM']

# Loop through unique cell types
#for celltype in ann_adata.obs['celltype'].unique():
for celltype in celltypes:
    # Subset the data for the celltype
    print(celltype)
    celltype_data = ann_adata[ann_adata.obs['celltype'] == celltype]
in:#general 
    # Perform the differential expression analysis
    sc.tl.rank_genes_groups(celltype_data, groupby='sample', reference='ctrl', method='t-test')
    
    # Store the results
    results_vs18[celltype] = celltype_data.uns['rank_genes_groups']
    
    # Now plot the results
    sc.pl.rank_genes_groups(celltype_data, n_genes=20)

    sc.pl.rank_genes_groups_dotplot(celltype_data, n_genes=20)

    for group in celltype_data.uns['rank_genes_groups']['names'].dtype.names:
        top_genes = celltype_data.uns['rank_genes_groups']['names'][group][:10]
        sc.pl.violin(celltype_data, keys=top_genes, groupby='sample')
    
    
# Now `results is a dictionary with cell types as keys. The values are the differential expression results for that cell type.
#results

### PODO

In [ ]:
# Select the subset for a specific cell type
adata_podo = ann_adata[ann_adata.obs['celltype'] == 'PODO']

# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(adata_podo, groupby='sample', reference='ctrl', method='t-test')

# Now plot the results
sc.pl.rank_genes_groups(adata_podo, n_genes=20)
sc.pl.rank_genes_groups_dotplot(adata_podo, n_genes=20)

for group in adata_podo.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = adata_podo.uns['rank_genes_groups']['names'][group][:10]
    sc.pl.violin(adata_podo, keys=top_genes, groupby='sample')

# Extract differential expression results
#de_results = adata_podo.uns['rank_genes_groups']

In [ ]:
p = 0.05
log2fc = 0

deg_podo_sg20 = sc.get.rank_genes_groups_df(adata_podo, group = 'age')

# significant genes -> filtered by pval and log2fc
deg_podo_sig_sg20 = deg_podo_sg20 [(deg_podo_sg20 ['pvals_adj'] < p) & (deg_podo_sg20 ['logfoldchanges'].abs() > log2fc)]
deg_podo_sig_up_sg20 = deg_podo_sg20 [(deg_podo_sg20 ['pvals_adj'] < p) & (deg_podo_sg20 ['logfoldchanges'] > 0 )]
deg_podo_sig_dwn_sg20 = deg_podo_sg20 [(deg_podo_sg20 ['pvals_adj'] < p) & (deg_podo_sg20 ['logfoldchanges'] < 0)]

print(deg_podo_sg20.shape)
print(deg_podo_sig_sg20.shape)
print(deg_podo_sig_up_sg20.shape)
print(deg_podo_sig_dwn_sg20.shape)

#deg_podo_sig_sg20.to_csv('./tables/degs_podo_AgevsCtrl.csv', index=False) 

#sc.pl.DotPlot(adata_podo_sg20, use_raw=True, var_names =deg_podo_sig_sg20.names, groupby='sample', standard_scale="var", title=f"Myo DEGs ctrl vs. aging", cmap='BuGn', figsize=(300, 5)).savefig('./dotplot.pdf' ,format='pdf')
GeneExpression.volcano(deg_podo_sg20 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.05, 0.05), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')

In [ ]:
# Extract the results into a DataFrame
degs = sc.get.rank_genes_groups_df(adata_podo, group='age')

# Filter for upregulated genes
# Assuming that `logfoldchanges` column indicates the direction of change
upregulated_genes = degs[degs['logfoldchanges'] > 0]['names'].tolist()

# Subset the AnnData object to only include upregulated genes
adata_podo_up = adata_podo[:, upregulated_genes]

# Plot heatmap of the upregulated genes
sc.pl.heatmap(adata_podo, var_names=upregulated_genes, groupby='sample', use_raw = True)

In [ ]:
from itertools import chain

#top_genes = adata_podo.uns['rank_genes_groups']['names'][::1][:100]
#top_genes_flattened = list(chain(*top_genes))

podo_genes = adata_podo.uns['rank_genes_groups']['names']

deg_podo_sg20_up = deg_podo_sg20[deg_podo_sg20.logfoldchanges>0 & (deg_podo_sg20.pvals_adj<0.05)]
deg_podo_sg20_dw = deg_podo_sg20[deg_podo_sg20.logfoldchanges<0 & (deg_podo_sg20.pvals_adj<0.05)]

sc.pl.heatmap(adata_podo, var_names = deg_podo_sg20_up.names, groupby = 'sample', use_raw = True#, swap_axes=True
             )
sc.pl.heatmap(adata_podo, var_names = deg_podo_sg20_dw.names, groupby = 'sample', use_raw = True#, swap_axes=True
             )
sc.pl.heatmap(adata_podo, var_names = top_genes_flattened, groupby = 'sample', use_raw = True#, swap_axes=True
             )

### EC1 (gEC)

In [ ]:
# Select the subset for a specific cell type
celltype_data = ann_adata[ann_adata.obs['celltype'] == 'EC-1(gEC)']

# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(celltype_data, groupby='sample', reference='ctrl', method='wilcoxon')

# Now plot the results
sc.pl.rank_genes_groups(celltype_data, n_genes=20)

sc.pl.rank_genes_groups_dotplot(celltype_data, n_genes=20)

for group in celltype_data.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = celltype_data.uns['rank_genes_groups']['names'][group][:10]
    sc.pl.violin(celltype_data, keys=top_genes, groupby='sample')

### vSMC/MC

In [ ]:
# Select the subset for a specific cell type
celltype_data = ann_adata[ann_adata.obs['celltype'] == 'vSMC/MC']

# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(celltype_data, groupby='sample', reference='ctrl', method='wilcoxon')

# Now plot the results
sc.pl.rank_genes_groups(celltype_data, n_genes=20)

sc.pl.rank_genes_groups_dotplot(celltype_data, n_genes=20)

for group in celltype_data.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = celltype_data.uns['rank_genes_groups']['names'][group][:10]
    sc.pl.violin(celltype_data, keys=top_genes, groupby='sample')


### PEC

### PT-1

In [ ]:
# Select the subset for a specific cell type
celltype_data = ann_adata[ann_adata.obs['celltype'] == 'PT-1']

# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(celltype_data, groupby='sample', reference='ctrl', method='wilcoxon')

# Now plot the results
sc.pl.rank_genes_groups(celltype_data, n_genes=20)

sc.pl.rank_genes_groups_dotplot(celltype_data, n_genes=20)

for group in celltype_data.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = celltype_data.uns['rank_genes_groups']['names'][group][:10]
    sc.pl.violin(celltype_data, keys=top_genes, groupby='sample')


### PT-2

In [ ]:
# Select the subset for a specific cell type
celltype_data = ann_adata[ann_adata.obs['celltype'] == 'PT-2']

# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(celltype_data, groupby='sample', reference='ctrl', method='wilcoxon')

# Now plot the results
sc.pl.rank_genes_groups(celltype_data, n_genes=20)

sc.pl.rank_genes_groups_dotplot(celltype_data, n_genes=20)

for group in celltype_data.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = celltype_data.uns['rank_genes_groups']['names'][group][:10]
    sc.pl.violin(celltype_data, keys=top_genes, groupby='sample')


### IMM

In [ ]:
# Select the subset for a specific cell type
celltype_data = ann_adata[ann_adata.obs['celltype'] == 'IMM']

# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(celltype_data, groupby='sample', reference='ctrl', method='wilcoxon')

# Now plot the results
sc.pl.rank_genes_groups(celltype_data, n_genes=20)

sc.pl.rank_genes_groups_dotplot(celltype_data, n_genes=20)

for group in celltype_data.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = celltype_data.uns['rank_genes_groups']['names'][group][:10]
    sc.pl.violin(celltype_data, keys=top_genes, groupby='sample')


## Differential Expression Analysis (Reference SG20)

In [ ]:
# Create a place to store results
results_vs20 = {}
# Specify the cell types of interest
celltypes = ['PODO', 'EC-1(gEC)', 'vSMC/MC', 
                         #'PEC',
                         'PT-1', 'PT-2', 'IMM']

# Loop through unique cell types
#for celltype in ann_adata.obs['celltype'].unique():
for celltype in celltypes:
    # Subset the data for the celltype
    print(celltype)
    celltype_data = ann_adata[ann_adata.obs['celltype'] == celltype]

    # Perform the differential expression analysis
    sc.tl.rank_genes_groups(celltype_data, groupby='sample', reference='age', method='t-test')
    
    # Store the results
    results_vs20[celltype] = celltype_data.uns['rank_genes_groups']
    
    # Now plot the results
    sc.pl.rank_genes_groups(celltype_data, n_genes=20)

    sc.pl.rank_genes_groups_dotplot(celltype_data, n_genes=20)

    for group in celltype_data.uns['rank_genes_groups']['names'].dtype.names:
        top_genes = celltype_data.uns['rank_genes_groups']['names'][group][:10]
        sc.pl.violin(celltype_data, keys=top_genes, groupby='sample')
    
    
# Now `results` is a dictionary with cell types as keys. The values are the differential expression results for that cell type.
#results_vs20

### PODO

In [ ]:
# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(adata_podo, groupby='sample', reference='age', method='t-test')

# Now plot the results
sc.pl.rank_genes_groups(adata_podo, n_genes=20)
sc.pl.rank_genes_groups_dotplot(adata_podo, n_genes=20)

for group in adata_podo.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = adata_podo.uns['rank_genes_groups']['names'][group][:10]
    sc.pl.violin(adata_podo, keys=top_genes, groupby='sample')

# Extract differential expression results
de_results = adata_podo.uns['rank_genes_groups']

In [ ]:
p = 0.05
log2fc = 0 

deg_podo_sg24 = sc.get.rank_genes_groups_df(adata_podo, group = 'SG24')
deg_podo_sg26 = sc.get.rank_genes_groups_df(adata_podo, group = 'SG26')
deg_podo_sg28 = sc.get.rank_genes_groups_df(adata_podo, group = 'SG28')

# significant genes -> filtered by pval and log2fc
deg_podo_sig_sg24 = deg_podo_sg24[(deg_podo_sg24['pvals_adj'] < p) & (deg_podo_sg24['logfoldchanges'].abs() > log2fc)]
deg_podo_sig_sg26 = deg_podo_sg26[(deg_podo_sg26['pvals_adj'] < p) & (deg_podo_sg26['logfoldchanges'].abs() > log2fc)]
deg_podo_sig_sg28 = deg_podo_sg28[(deg_podo_sg28['pvals_adj'] < p) & (deg_podo_sg28['logfoldchanges'].abs() > log2fc)]

deg_podo_sig_up_sg24 = deg_podo_sg24[(deg_podo_sg24['pvals_adj'] < p) & (deg_podo_sg24['logfoldchanges'] > 0 )]
deg_podo_sig_up_sg26 = deg_podo_sg26[(deg_podo_sg26['pvals_adj'] < p) & (deg_podo_sg26['logfoldchanges'] > 0 )]
deg_podo_sig_up_sg28 = deg_podo_sg28[(deg_podo_sg28['pvals_adj'] < p) & (deg_podo_sg28['logfoldchanges'] > 0 )]

deg_podo_sig_dwn_sg24 = deg_podo_sg24[(deg_podo_sg24['pvals_adj'] < p) & (deg_podo_sg24['logfoldchanges'] < 0)]
deg_podo_sig_dwn_sg26 = deg_podo_sg26[(deg_podo_sg26['pvals_adj'] < p) & (deg_podo_sg26['logfoldchanges'] < 0)]
deg_podo_sig_dwn_sg28 = deg_podo_sg28[(deg_podo_sg28['pvals_adj'] < p) & (deg_podo_sg28['logfoldchanges'] < 0)]

print('sg24: ', deg_podo_sg24.shape)
print(deg_podo_sig_sg24.shape)
print(deg_podo_sig_up_sg24.shape)
print(deg_podo_sig_dwn_sg24.shape)

print('sg26: ', deg_podo_sg26.shape)
print(deg_podo_sig_sg26.shape)
print(deg_podo_sig_up_sg26.shape)
print(deg_podo_sig_dwn_sg26.shape)

print('sg28: ', deg_podo_sg28.shape)
print(deg_podo_sig_sg28.shape)
print(deg_podo_sig_up_sg28.shape)
print(deg_podo_sig_dwn_sg28.shape)

#deg_podo_sig_sg24.to_csv('./sig_degs/degs_podo_aging_sAct.csv', index=False)
#deg_podo_sig_sg26.to_csv('./sig_degs/degs_podo_aging_DR.csv', index=False)
#deg_podo_sig_sg28.to_csv('./sig_degs/degs_podo_aging_comb.csv', index=False) 

GeneExpression.volcano(deg_podo_sg24 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.05, 0.05), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')
GeneExpression.volcano(deg_podo_sg26 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.05, 0.05), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')
GeneExpression.volcano(deg_podo_sg28 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.05, 0.05), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')

### EC1

In [ ]:
# Select the subset for a specific cell type
celltype_data = ann_adata[ann_adata.obs['celltype'] == 'EC-1(gEC)']

# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(celltype_data, groupby='sample', reference='age', method='wilcoxon')

# Now plot the results
sc.pl.rank_genes_groups(celltype_data, n_genes=20)

sc.pl.rank_genes_groups_dotplot(celltype_data, n_genes=20)

for group in celltype_data.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = celltype_data.uns['rank_genes_groups']['names'][group][:10]
    sc.pl.violin(celltype_data, keys=top_genes, groupby='sample')

### vSMC/MC

In [ ]:
# Select the subset for a specific cell type
celltype_data = ann_adata[ann_adata.obs['celltype'] == 'vSMC/MC']

# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(celltype_data, groupby='sample', reference='age', method='wilcoxon')

# Now plot the results
sc.pl.rank_genes_groups(celltype_data, n_genes=20)

sc.pl.rank_genes_groups_dotplot(celltype_data, n_genes=20)

for group in celltype_data.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = celltype_data.uns['rank_genes_groups']['names'][group][:10]
    sc.pl.violin(celltype_data, keys=top_genes, groupby='sample')

### PEC

### PT-1

In [ ]:
# Select the subset for a specific cell type
celltype_data = ann_adata[ann_adata.obs['celltype'] == 'PT-1']

# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(celltype_data, groupby='sample', reference='age', method='wilcoxon')

# Now plot the results
sc.pl.rank_genes_groups(celltype_data, n_genes=20)

sc.pl.rank_genes_groups_dotplot(celltype_data, n_genes=20)

for group in celltype_data.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = celltype_data.uns['rank_genes_groups']['names'][group][:10]
    sc.pl.violin(celltype_data, keys=top_genes, groupby='sample')

### PT-2

In [ ]:
# Select the subset for a specific cell type
celltype_data = ann_adata[ann_adata.obs['celltype'] == 'PT-2']

# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(celltype_data, groupby='sample', reference='age', method='wilcoxon')

# Now plot the results
sc.pl.rank_genes_groups(celltype_data, n_genes=20)

sc.pl.rank_genes_groups_dotplot(celltype_data, n_genes=20)

for group in celltype_data.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = celltype_data.uns['rank_genes_groups']['names'][group][:10]
    sc.pl.violin(celltype_data, keys=top_genes, groupby='sample')

### IMM

In [ ]:
# Select the subset for a specific cell type
celltype_data = ann_adata[ann_adata.obs['celltype'] == 'IMM']

# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(celltype_data, groupby='sample', reference='age', method='wilcoxon')

# Now plot the results
sc.pl.rank_genes_groups(celltype_data, n_genes=20)

sc.pl.rank_genes_groups_dotplot(celltype_data, n_genes=20)

for group in celltype_data.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = celltype_data.uns['rank_genes_groups']['names'][group][:10]
    sc.pl.violin(celltype_data, keys=top_genes, groupby='sample')

## Intersection

In [ ]:
#print("\033[1m" + 'Regulated by aging: '+ "\033[0m" ,deg_podo_sig_sg20)
print("\033[1m" + 'Upregulated by aging: '+ "\033[0m" ,deg_podo_sig_up_sg20)
print("\033[1m" + 'Downregulated by aging: '+ "\033[0m",deg_podo_sig_dwn_sg20)

print("\033[1m" + 'Regulated by sAct: '+ "\033[0m" , deg_podo_sig_sg24)
#print("\033[1m" + 'Upregulated by sAct: '+ "\033[0m" , deg_podo_sig_up_sg24)
#print("\033[1m" + 'Downregulated by sAct: '+ "\033[0m" , deg_podo_sig_dwn_sg24)

print("\033[1m" + 'Regulated by DR: '+ "\033[0m" , deg_podo_sig_sg26)
#print("\033[1m" + 'Upregulated by DR: '+ "\033[0m" , deg_podo_sig_up_sg26)
#print("\033[1m" + 'Downregulated by DR: '+ "\033[0m" , deg_podo_sig_dwn_sg26)

print("\033[1m" + 'Regulated by combi: '+ "\033[0m" , deg_podo_sig_sg28)
#print("\033[1m" + 'Upregulated by combi: '+ "\033[0m" , deg_podo_sig_up_sg28)
#print("\033[1m" + 'Downregulated by combi: '+ "\033[0m" , deg_podo_sig_dwn_sg28)

### Aging and sAct

In [ ]:
# Use merge on aging up - rescued by sAct 
result = pd.merge(deg_podo_sig_up_sg20, deg_podo_sig_sg24, on='names', suffixes=('_age', '_treat'))
print(result.shape)
# Assuming you have the result data frame from the previous step
degs_podo_ageup_sAct = result[result['logfoldchanges_age'] > result['logfoldchanges_treat']]

#degs_podo_ageup_sAct.to_csv('./tables/degs_podo_ageup_sAct.csv', index=False) 
print(degs_podo_ageup_sAct.shape)
print(degs_podo_ageup_sAct.head())

# Create a bar plot for log-fold changes
plt.figure(figsize=(30, 5))  # Adjust the figure size as needed

# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors_age = ['green' if x >= 0 else 'red' for x in  degs_podo_ageup_sAct['logfoldchanges_age']]
colors_treat = ['blue' if x >= 0 else 'orange' for x in  degs_podo_ageup_sAct['logfoldchanges_treat']]

# Plot the bar for logfoldchanges_age
plt.bar(degs_podo_ageup_sAct['names'], degs_podo_ageup_sAct['logfoldchanges_age'], color=colors_age, width=0.4, align='center', label='logfoldchanges_age')

# Plot the bar for logfoldchanges_treat next to the first bar
plt.bar(degs_podo_ageup_sAct['names'], degs_podo_ageup_sAct['logfoldchanges_treat'], color=colors_treat, width=0.4, align='edge', label='logfoldchanges_treat')
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)  
plt.legend()
# Set margin
plt.subplots_adjust(bottom=0.4)
    
# Save or display the plot (customize the path as needed)
plt.savefig(f'logfoldchanges_podo_ageup_sAct.png')
plt.show() 

In [ ]:
# Use merge on aging dwn - rescued by sAct (sg24)
result = pd.merge(deg_podo_sig_dwn_sg20, deg_podo_sig_sg24, on='names', suffixes=('_age', '_treat'))
print(result.shape)
degs_podo_agedwn_sAct = result[result['logfoldchanges_age'] < result['logfoldchanges_treat']]

#degs_podo_agedwn_sAct.to_csv('./tables/degs_podo_agedwn_sAct.csv', index=False) 
print(degs_podo_agedwn_sAct.head())

#result
# Create a bar plot for log-fold changes
plt.figure(figsize=(250, 5))  # Adjust the figure size as needed

# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors_age = ['green' if x >= 0 else 'red' for x in  degs_podo_agedwn_sAct['logfoldchanges_age']]
colors_treat = ['blue' if x >= 0 else 'orange' for x in  degs_podo_agedwn_sAct['logfoldchanges_treat']]

# Plot the bar for logfoldchanges_age
plt.bar(degs_podo_agedwn_sAct['names'], degs_podo_agedwn_sAct['logfoldchanges_age'], color=colors_age, width=0.4, align='center', label='logfoldchanges_age')

# Plot the bar for logfoldchanges_treat next to the first bar
plt.bar(degs_podo_agedwn_sAct['names'], degs_podo_agedwn_sAct['logfoldchanges_treat'], color=colors_treat, width=0.4, align='edge', label='logfoldchanges_treat')
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)  # Add a legend 
plt.legend()
# Set margin
plt.subplots_adjust(bottom=0.4)
    
# Save or display the plot (customize the path as needed)
plt.savefig(f'logfoldchanges_podo_agedwn_sAct.png')
plt.show() 

### Aging and DR

In [ ]:
# Use merge on aging up - rescued by DR 
result = pd.merge(deg_podo_sig_up_sg20, deg_podo_sig_sg26, on='names', suffixes=('_age', '_treat'))
print(result.shape)
# Assuming you have the result data frame from the previous step
degs_podo_ageup_DR = result[result['logfoldchanges_age'] > result['logfoldchanges_treat']]

#degs_podo_ageup_DR.to_csv('./tables/degs_podo_ageup_DR.csv', index=False) 
print(degs_podo_ageup_DR.shape)
print(degs_podo_ageup_DR.head())

# Create a bar plot for log-fold changes
plt.figure(figsize=(70, 5))  # Adjust the figure size as needed

# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors_age = ['green' if x >= 0 else 'red' for x in  degs_podo_ageup_DR['logfoldchanges_age']]
colors_treat = ['blue' if x >= 0 else 'orange' for x in  degs_podo_ageup_DR['logfoldchanges_treat']]

# Plot the bar for logfoldchanges_age
plt.bar(degs_podo_ageup_DR['names'], degs_podo_ageup_DR['logfoldchanges_age'], color=colors_age, width=0.4, align='center', label='logfoldchanges_age')

# Plot the bar for logfoldchanges_treat next to the first bar
plt.bar(degs_podo_ageup_DR['names'], degs_podo_ageup_DR['logfoldchanges_treat'], color=colors_treat, width=0.4, align='edge', label='logfoldchanges_treat')
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)
plt.legend()
# Set margin
plt.subplots_adjust(bottom=0.4)
    
# Save or display the plot (customize the path as needed)
plt.savefig(f'logfoldchanges_podo_ageup_DR.png')
plt.show() 

In [ ]:
# Use merge on aging dwn - rescued by DR
result = pd.merge(deg_podo_sig_dwn_sg20, deg_podo_sig_sg26, on='names', suffixes=('_age', '_treat'))
print(result.shape)
degs_podo_agedwn_DR = result[result['logfoldchanges_age'] < result['logfoldchanges_treat']]
#result

#degs_podo_agedwn_DR.to_csv('./tables/degs_podo_agedwn_DR.csv', index=False) 
print(degs_podo_agedwn_DR.shape)
print(degs_podo_agedwn_DR.head())

# Create a bar plot for log-fold changes
plt.figure(figsize=(150, 5))  # Adjust the figure size as needed

# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors_age = ['green' if x >= 0 else 'red' for x in  degs_podo_agedwn_DR['logfoldchanges_age']]
colors_treat = ['blue' if x >= 0 else 'orange' for x in  degs_podo_agedwn_DR['logfoldchanges_treat']]

# Plot the bar for logfoldchanges_age
plt.bar(degs_podo_agedwn_DR['names'], degs_podo_agedwn_DR['logfoldchanges_age'], color=colors_age, width=0.4, align='center', label='logfoldchanges_age')

# Plot the bar for logfoldchanges_treat next to the first bar
plt.bar(degs_podo_agedwn_DR['names'], degs_podo_agedwn_DR['logfoldchanges_treat'], color=colors_treat, width=0.4, align='edge', label='logfoldchanges_treat')
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)  
plt.legend()

# Set margin
plt.subplots_adjust(bottom=0.4)
    
# Save or display the plot (customize the path as needed)
plt.savefig(f'logfoldchanges_podo_agedwn_DR.png')
plt.show() 

### Aging and combi

In [ ]:
# Use merge on aging up - rescued by combi
result = pd.merge(deg_podo_sig_up_sg20, deg_podo_sig_sg28, on='names', suffixes=('_age', '_treat'))
print(result.shape)

# Assuming you have the result data frame from the previous step
degs_podo_ageup_comb = result[result['logfoldchanges_age'] > result['logfoldchanges_treat']]
#degs_podo_ageup_comb.to_csv('./tables/degs_podo_ageup_comb.csv', index=False) 
print(degs_podo_ageup_comb.shape)
print(degs_podo_ageup_comb.head())

# Create a bar plot for log-fold changes
plt.figure(figsize=(50, 5))  # Adjust the figure size as needed

# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors_age = ['green' if x >= 0 else 'red' for x in  degs_podo_ageup_comb['logfoldchanges_age']]
colors_treat = ['blue' if x >= 0 else 'orange' for x in  degs_podo_ageup_comb['logfoldchanges_treat']]

# Plot the bar for logfoldchanges_age
plt.bar(degs_podo_ageup_comb['names'], degs_podo_ageup_comb['logfoldchanges_age'], color=colors_age, width=0.4, align='center', label='logfoldchanges_age')

# Plot the bar for logfoldchanges_treat next to the first bar
plt.bar(degs_podo_ageup_comb['names'], degs_podo_ageup_comb['logfoldchanges_treat'], color=colors_treat, width=0.4, align='edge', label='logfoldchanges_treat')
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)
plt.legend()
# Set margin
plt.subplots_adjust(bottom=0.4)
    
# Save or display the plot (customize the path as needed)
plt.savefig(f'logfoldchanges_podo_ageup_comb.png')
plt.show() 

In [ ]:
# Use merge on aging dwn - rescued by combi
result = pd.merge(deg_podo_sig_dwn_sg20, deg_podo_sig_sg28, on='names', suffixes=('_age', '_treat'))
print(result.shape)

degs_podo_agedwn_comb = result[result['logfoldchanges_age'] < result['logfoldchanges_treat']]
#degs_podo_agedwn_comb.to_csv('./tables/degs_podo_agedwn_comb.csv', index=False) 
print(degs_podo_agedwn_comb.shape)
print(degs_podo_agedwn_comb.head())

# Create a bar plot for log-fold changes
plt.figure(figsize=(50, 5))  # Adjust the figure size as needed

# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors_age = ['green' if x >= 0 else 'red' for x in  degs_podo_agedwn_comb['logfoldchanges_age']]
colors_treat = ['blue' if x >= 0 else 'orange' for x in  degs_podo_agedwn_comb['logfoldchanges_treat']]

# Plot the bar for logfoldchanges_age
plt.bar(degs_podo_agedwn_comb['names'], degs_podo_agedwn_comb['logfoldchanges_age'], color=colors_age, width=0.4, align='center', label='logfoldchanges_age')

# Plot the bar for logfoldchanges_treat next to the first bar
plt.bar(degs_podo_agedwn_comb['names'], degs_podo_agedwn_comb['logfoldchanges_treat'], color=colors_treat, width=0.4, align='edge', label='logfoldchanges_treat')
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)
plt.legend()
# Set margin
plt.subplots_adjust(bottom=0.4)

# Save or display the plot (customize the path as needed)
plt.savefig(f'logfoldchanges_podo_agedwn_comb.png')
plt.show() 

## Gene Set Enrichment Analysis (GSEA)

In [ ]:
from gseapy.scipalette import SciPalette
NbDr = SciPalette.create_colormap() 

### Aging in Podocytes

In [ ]:
enr_up = gp.enrichr(deg_podo_sig_up_sg20.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_dw = gp.enrichr(deg_podo_sig_dwn_sg20.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_up_go = gp.enrichr(deg_podo_sig_up_sg20.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

enr_dw_go = gp.enrichr(deg_podo_sig_dwn_sg20.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

# trim (go:...)
enr_up_go.res2d.Term = enr_up_go.res2d.Term.str.split(" \(GO").str[0]
enr_dw_go.res2d.Term = enr_dw_go.res2d.Term.str.split(" \(GO").str[0]

gp.dotplot(enr_up.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Upregulated by aging in Podocytes", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Downregulated by aging in Podocytes", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

gp.dotplot(enr_up_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Upregulated by aging in Podocytes", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Downregulated by aging in Podocytes", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

### Aging and sAct

In [ ]:
enr_up = gp.enrichr(degs_podo_agedwn_sAct.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_dw = gp.enrichr(degs_podo_ageup_sAct.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_up_go = gp.enrichr(degs_podo_agedwn_sAct.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
enr_dw_go = gp.enrichr(degs_podo_ageup_sAct.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

# trim (go:...)
enr_up_go.res2d.Term = enr_up_go.res2d.Term.str.split(" \(GO").str[0]
enr_dw_go.res2d.Term = enr_dw_go.res2d.Term.str.split(" \(GO").str[0]

gp.dotplot(enr_up.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Up in aging, rescued by sAct in podocytes", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Down in aging, rescued by sAct in podocytes", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

gp.dotplot(enr_up_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Up in aging, rescued by sAct in podocytes", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Down in aging, rescued by sAct in podocytes", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

### Aging and DR

In [ ]:
enr_up = gp.enrichr(degs_podo_agedwn_DR.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_dw = gp.enrichr(degs_podo_ageup_DR.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_up_go = gp.enrichr(degs_podo_agedwn_DR.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
enr_dw_go = gp.enrichr(degs_podo_ageup_DR.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

# trim (go:...)
enr_up_go.res2d.Term = enr_up_go.res2d.Term.str.split(" \(GO").str[0]
enr_dw_go.res2d.Term = enr_dw_go.res2d.Term.str.split(" \(GO").str[0]

gp.dotplot(enr_up.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Up in aging, rescued by DR in podocytes", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Down in aging, rescued by DR in podocytes", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

gp.dotplot(enr_up_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Up in aging, rescued by DR in podocytes", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Down in aging, rescued by DR in podocytes", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

### Aging and combi

In [ ]:
enr_up = gp.enrichr(degs_podo_agedwn_comb.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_dw = gp.enrichr(degs_podo_ageup_comb.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_up_go = gp.enrichr(degs_podo_agedwn_comb.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
enr_dw_go = gp.enrichr(degs_podo_ageup_comb.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

# trim (go:...)
enr_up_go.res2d.Term = enr_up_go.res2d.Term.str.split(" \(GO").str[0]
enr_dw_go.res2d.Term = enr_dw_go.res2d.Term.str.split(" \(GO").str[0]

gp.dotplot(enr_up.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Up in aging, rescued by combi in podocytes", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Down in aging, rescued by combi in podocytes", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

gp.dotplot(enr_up_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Up in aging, rescued by combi in podocytes", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Down in aging, rescued by combi in podocytes", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

## Cell-Cell Interaction Analysis

In [ ]:
# Map mouse to human genes
hum_ann_adata = translate(ann_adata)

In [ ]:
hum_ann_adata.obs

In [ ]:
# Define the file paths
meta_file_path = '/data/projects/manuela/scRNA_aging-mouse/meta.txt'
counts_file_path = '/data/projects/manuela/scRNA_aging-mouse/counts.txt'

output_path = '/data/projects/manuela/scRNA_aging-mouse/cellphonedb'
cpdb = '/data/projects/manuela/scRNA_aging-mouse/cellphonedb/v4.1.0/cellphonedb.zip'

# Save metadata 
meta = hum_ann_adata.obs
meta = meta[['celltype']]
# Set Index Name
meta.index.name='Cell'

# Save counts 
counts = pd.DataFrame(data = hum_ann_adata.X,  # use raw counts directly
                      index = hum_ann_adata.obs_names,  # use cell names as index
                      columns = hum_ann_adata.var_names)  # use gene names as columns

counts = counts.T
# Set Index Name
counts.index.name='Gene'

# Save metadata and counts as CSV
#meta.to_csv(meta_file_path, sep = '\t')
#counts.to_csv(counts_file_path, sep = '\t')
#hum_ann_adata.write('counts.h5ad')

In [ ]:
# Run the CellPhoneDB analysis
deconvoluted, means, pvalues, significant_means = cpdb_statistical_analysis_method.call(
        cpdb_file_path = cpdb, 
        meta_file_path = '/data/projects/manuela/scRNA_aging-mouse/meta.txt',
        counts_file_path = '/data/projects/manuela/scRNA_aging-mouse/counts.h5ad',
        counts_data = 'hgnc_symbol',   # specify that you are using gene symbols
       # counts_data = 'ensembl',   
        output_path = output_path)

In [ ]:
print(deconvoluted.head(3))
print(means.head(3))
print(pvalues.head(3))
print(significant_means.head(3))

In [ ]:
search_results = search_utils.search_analysis_results(
    query_cell_types_1 = ['PODO', 'vSMC/MC'],  # List of cells 1, will be paired to cells 2 (list or 'All').
    query_cell_types_2 = ['vSMC/MC', 'PODO'],     # List of cells 2, will be paired to cells 1 (list or 'All').
   # query_genes = ['TGFBR1'],                                       # filter interactions based on the genes participating (list).
 #   query_interactions = ['CSF1_CSF1R'],                            # filter intereactions based on their name (list).
    significant_means = significant_means,                          # significant_means file generated by CellPhoneDB.
    deconvoluted = deconvoluted,                                    # devonvoluted file generated by CellPhoneDB.
    separator = '|',                                                # separator (default: |) employed to split cells (cellA|cellB).
    long_format = True                                              # converts the output into a wide table, removing non-significant interactions
)

search_results

In [ ]:
search_results = search_utils.search_analysis_results(
    query_cell_types_1 = ['PODO', 'EC-1(gEC)', 'vSMC/MC', 'PEC', 'PT-1', 'PT-2', 'IMM'],  # List of cells 1, will be paired to cells 2 (list or 'All').
    query_cell_types_2 = ['PODO', 'EC-1(gEC)', 'vSMC/MC', 'PEC', 'PT-1', 'PT-2', 'IMM'],     # List of cells 2, will be paired to cells 1 (list or 'All').
   # query_genes = ['TGFBR1'],                                       # filter interactions based on the genes participating (list).
 #   query_interactions = ['CSF1_CSF1R'],                            # filter intereactions based on their name (list).
    significant_means = significant_means,                          # significant_means file generated by CellPhoneDB.
    deconvoluted = deconvoluted,                                    # devonvoluted file generated by CellPhoneDB.
    separator = '|',                                                # separator (default: |) employed to split cells (cellA|cellB).
    long_format = True                                              # converts the output into a wide table, removing non-significant interactions
)

search_results      

### Basic analysis and plotting

### Basic Analysis and plotting of each single sample against reference (SG18) 

In [ ]:
# Get the unique samples
samples = hum_ann_adata.obs['sample'].unique()

# Get the reference sample
reference_sample = hum_ann_adata[hum_ann_adata.obs['sample'] == 'SG18']

# Loop over the samples
for sample in samples:
    if sample == 'SG18':  # Skip the reference sample
        continue

    # Get the current sample
    current_sample = hum_ann_adata[hum_ann_adata.obs['sample'] == sample]
    
    # Concatenate the current sample with the reference sample
    sample_data = current_sample.concatenate(reference_sample)
    
    # Save metadata 
    meta = sample_data.obs
    meta = meta[['celltype']]
    # Set Index Name
    meta.index.name='Cell'

    # Save metadata and counts as CSV
    #meta.to_csv(f'/data/projects/manuela/scRNA_aging-mouse/meta_{sample}.txt', sep = '\t')
    # Save counts as h5ad
    #sample_data.write(f'/data/projects/manuela/scRNA_aging-mouse/counts_{sample}.h5ad')

    # Run the CellPhoneDB analysis
    deconvoluted, means, pvalues, significant_means = cpdb_statistical_analysis_method.call(
            cpdb_file_path = cpdb, 
            meta_file_path = f'/data/projects/manuela/scRNA_aging-mouse/meta_{sample}.txt',
            counts_file_path = f'/data/projects/manuela/scRNA_aging-mouse/counts_{sample}.h5ad',
            counts_data = 'hgnc_symbol',   # specify that you are using gene symbols
           # counts_data = 'ensembl',   
            output_path = f'/data/projects/manuela/scRNA_aging-mouse/cellphonedb/{sample}_vs_SG18')

    # Rest of the code remains the same
    search_results = search_utils.search_analysis_results(
        query_cell_types_1 = ['PODO', 'vSMC/MC'],  # List of cells 1, will be paired to cells 2 (list or 'All').
        query_cell_types_2 = ['vSMC/MC', 'PODO'],     # List of cells 2, will be paired to cells 1 (list or 'All').
       # query_genes = ['TGFBR1'],                                       # filter interactions based on the genes participating (list).
     #   query_interactions = ['CSF1_CSF1R'],                            # filter intereactions based on their name (list).
        significant_means = significant_means,                          # significant_means file generated by CellPhoneDB.
        deconvoluted = deconvoluted,                                    # devonvoluted file generated by CellPhoneDB.
        separator = '|',                                                # separator (default: |) employed to split cells (cellA|cellB).
        long_format = True                                              # converts the output into a wide table, removing non-significant interactions
    )
    
    dotplot = kpy.plot_cpdb(
        adata = sample_data,
        cell_type1 = "PODO|vSMC/MC|PECs?|PT-2|IMM",
        # include EC-1(gEC)
        cell_type2 = "PODO|vSMC/MC|PECs?|PT-1", 
        means = means,
        pvals = pvalues,
        celltype_key = "celltype",
     #   genes = ["TGFB2", "CSF1R"],
        figsize = (15,5),
        title = f"Interactions vs. SG18 between PODO, vSMC/MC, PEC, PT-1, PT-2, IMM for {sample}",
        max_size = 6,
        highlight_size = 0.75,
        standard_scale = True
    )
    dotplot.save(f'/data/projects/manuela/scRNA_aging-mouse/cellphonedb/{sample}_vs_SG18/{sample}_dotplot.png')
    
    # Generate plots for the current sample
    heatmap = kpy.plot_cpdb_heatmap(
            adata = sample_data,
            pvals = pvalues,
            celltype_key = "celltype",
            figsize = (5,5),
            title = f"Number of significant interactions vs. SG18 for {sample}",
            symmetrical = False,
        )
    heatmap.figure.savefig(f'/data/projects/manuela/scRNA_aging-mouse/cellphonedb/{sample}_vs_SG18/{sample}_heatmap.png')

    # First create an empty graph
    G = nx.Graph()

    # Add nodes
    for gene in pd.concat([significant_means['gene_a'], significant_means['gene_b']]).unique():
        G.add_node(gene)

    # Add edges
    for i, row in significant_means.iterrows():
        G.add_edge(row['gene_a'], row['gene_b'])

    # Draw the graph
    plt.figure(figsize=(10,10))  # Set the figure size (optional)
    nx.draw(G, with_labels=True)
    plt.savefig(f'/data/projects/manuela/scRNA_aging-mouse/cellphonedb/{sample}_vs_SG18/{sample}_graph.png')
    plt.close()

### Basic Analysis and plotting of each single sample against reference (SG20) 

In [ ]:
# Get the unique samples
samples = hum_ann_adata.obs['sample'].unique()

# Get the reference sample
reference_sample = hum_ann_adata[hum_ann_adata.obs['sample'] == 'SG20']

# Loop over the samples
for sample in samples:
    if sample == 'SG20':  # Skip the reference sample
        continue

    # Get the current sample
    current_sample = hum_ann_adata[hum_ann_adata.obs['sample'] == sample]
    
    # Concatenate the current sample with the reference sample
    sample_data = current_sample.concatenate(reference_sample)
    
    # Save metadata 
    meta = sample_data.obs
    meta = meta[['celltype']]
    # Set Index Name
    meta.index.name='Cell'

    # Save metadata and counts as CSV
    #meta.to_csv(f'/data/projects/manuela/scRNA_aging-mouse/meta_{sample}.txt', sep = '\t')
    # Save counts as h5ad
    #sample_data.write(f'/data/projects/manuela/scRNA_aging-mouse/counts_{sample}.h5ad')

    # Run the CellPhoneDB analysis
    deconvoluted, means, pvalues, significant_means = cpdb_statistical_analysis_method.call(
            cpdb_file_path = cpdb, 
            meta_file_path = f'/data/projects/manuela/scRNA_aging-mouse/meta_{sample}.txt',
            counts_file_path = f'/data/projects/manuela/scRNA_aging-mouse/counts_{sample}.h5ad',
            counts_data = 'hgnc_symbol',   # specify that you are using gene symbols
           # counts_data = 'ensembl',   
            output_path = f'/data/projects/manuela/scRNA_aging-mouse/cellphonedb/{sample}_vs_SG20')

    # Rest of the code remains the same
    search_results = search_utils.search_analysis_results(
        query_cell_types_1 = ['PODO', 'vSMC/MC'],  # List of cells 1, will be paired to cells 2 (list or 'All').
        query_cell_types_2 = ['vSMC/MC', 'PODO'],     # List of cells 2, will be paired to cells 1 (list or 'All').
       # query_genes = ['TGFBR1'],                                       # filter interactions based on the genes participating (list).
     #   query_interactions = ['CSF1_CSF1R'],                            # filter intereactions based on their name (list).
        significant_means = significant_means,                          # significant_means file generated by CellPhoneDB.
        deconvoluted = deconvoluted,                                    # devonvoluted file generated by CellPhoneDB.
        separator = '|',                                                # separator (default: |) employed to split cells (cellA|cellB).
        long_format = True                                              # converts the output into a wide table, removing non-significant interactions
    )
    
    dotplot = kpy.plot_cpdb(
        adata = sample_data,
        cell_type1 = "PODO|vSMC/MC|PECs?|PT-2|IMM",
        # include EC-1(gEC)
        cell_type2 = "PODO|vSMC/MC|PECs?|PT-1", 
        means = means,
        pvals = pvalues,
        celltype_key = "celltype",
     #   genes = ["TGFB2", "CSF1R"],
        figsize = (15,5),
        title = f"Interactions vs. SG20 between PODO, vSMC/MC, PEC, PT-1, PT-2, IMM for {sample}",
        max_size = 6,
        highlight_size = 0.75,
        standard_scale = True
    )
    dotplot.save(f'/data/projects/manuela/scRNA_aging-mouse/cellphonedb/{sample}_vs_SG20/{sample}_dotplot.png')
    
    # Generate plots for the current sample
    heatmap = kpy.plot_cpdb_heatmap(
            adata = sample_data,
            pvals = pvalues,
            celltype_key = "celltype",
            figsize = (5,5),
            title = f"Number of significant interactions vs. SG20 for {sample}",
            symmetrical = False,
        )
    heatmap.figure.savefig(f'/data/projects/manuela/scRNA_aging-mouse/cellphonedb/{sample}_vs_SG20/{sample}_heatmap.png')
    
    # First create an empty graph
    G = nx.Graph()

    # Add nodes
    for gene in pd.concat([significant_means['gene_a'], significant_means['gene_b']]).unique():
        G.add_node(gene)

    # Add edges
    for i, row in significant_means.iterrows():
        G.add_edge(row['gene_a'], row['gene_b'])

    # Draw the graph
    plt.figure(figsize=(10,10))  # Set the figure size (optional)
    nx.draw(G, with_labels=True)
    plt.savefig(f'/data/projects/manuela/scRNA_aging-mouse/cellphonedb/{sample}_vs_SG20/{sample}_graph.png')
    plt.close()

### Freestyle Visualizations

In [ ]:
# Concatenate gene_a and gene_b columns
all_genes = pd.concat([significant_means['gene_a'], significant_means['gene_b']])

# Count the occurrences of each gene
gene_counts = all_genes.value_counts()

# Plot a bar chart
gene_counts[:20].plot(kind='bar')

plt.ylabel('Number of Interactions')
plt.title('Top 20 genes with most interactions')

# Rotate x-axis labels
plt.xticks(rotation=45, ha="right")  # Rotate labels by 45 degrees and align them right

# Add some space at the bottom of the plot
plt.tight_layout()

plt.show()


In [ ]:
# Get top 20 genes with most interactions
top_genes = gene_counts[:20].index

# Filter significant_means for rows where either gene_a or gene_b is in top_genes
top_interactions = significant_means[(significant_means['gene_a'].isin(top_genes)) | (significant_means['gene_b'].isin(top_genes))]

# Keep only the cell-type columns and compute the mean interaction score for each gene
interaction_heatmap = top_interactions.groupby('gene_a').mean()

# Plot a heatmap
plt.figure(figsize=(25,9))
sns.heatmap(interaction_heatmap, cmap="viridis")
plt.xticks(rotation=90, ha="right")  # Rotate x-axis labels 
#plt.tight_layout()  # Adjust the layout
plt.subplots_adjust(bottom=0.25) 

plt.show()


In [ ]:
# List of cell types to include
cell_types_to_include = ["PODO", "vSMC/MC", "PECs?", "PT-2", "IMM", "EC-1(gEC)"]

# Generate all combinations of cell types
cell_type_combinations = list(itertools.combinations(cell_types_to_include, 2))

# Convert tuples to strings with '|' separator
cell_type_combinations = ['|'.join(combination) for combination in cell_type_combinations]

# Filter significant_means for rows where either gene_a or gene_b is in top_genes
top_interactions = significant_means[(significant_means['gene_a'].isin(top_genes)) | (significant_means['gene_b'].isin(top_genes))]

# Group by gene_a and calculate the mean interaction score, then filter columns to include only selected cell types
interaction_heatmap = top_interactions.groupby('gene_a').mean()[cell_type_combinations]

# Plot a heatmap
plt.figure(figsize=(25,9))
sns.heatmap(interaction_heatmap, cmap="viridis")
plt.xticks(rotation=90, ha="right")  # Rotate x-axis labels 
plt.subplots_adjust(bottom=0.25) 

plt.show()


## Analyse Networks (networkX)

In [ ]:
significant_means.head()

In [ ]:
# First create an empty graph
G = nx.Graph()

# Add nodes
for gene in pd.concat([significant_means['gene_a'], significant_means['gene_b']]).unique():
    G.add_node(gene)

# Add edges
for i, row in significant_means.iterrows():
    G.add_edge(row['gene_a'], row['gene_b'])

# Draw the graph
nx.draw(G, with_labels=True)
plt.show()

In [ ]:
!pip freeze